Self-Healing Data Pipeline — auto-detects anomalies, classifies severity with Cortex AI, and triggers remediation.
*Co-authored with CoCo*

# Self-Healing Data Pipeline

An autonomous data pipeline that:
1. **Ingests** data into landing tables
2. **Monitors** data quality with automated checks
3. **Detects** anomalies via streams
4. **Classifies** issues using Cortex AI (severity + root cause)
5. **Self-heals** by applying automated remediation
6. **Escalates** to humans only when auto-fix fails

---

## Architecture
```
[Source Data] → [Landing Table] → [Quality Checks]
                                         ↓
                              [Anomaly Stream] → [AI Classifier]
                                                       ↓
                                              [Auto-Remediation] → [Escalation/Alert]
                                                       ↓
                                              [Audit Log + Dashboard]
```

## 1. Infrastructure Setup
Create the database, schemas, and warehouse for the pipeline.

In [1]:
%%sql -r infra_result
-- Create dedicated database and schemas
CREATE DATABASE IF NOT EXISTS SELF_HEALING_PIPELINE;

USE DATABASE SELF_HEALING_PIPELINE;

-- Schema for raw/landing data
CREATE SCHEMA IF NOT EXISTS RAW;
-- Schema for quality monitoring objects
CREATE SCHEMA IF NOT EXISTS MONITORING;
-- Schema for remediation and audit
CREATE SCHEMA IF NOT EXISTS OPERATIONS;

SELECT 'Infrastructure created successfully' AS status;

UsageError: Cell magic `%%sql` not found.


## 2. Ingestion Layer
Create sample landing tables that simulate a real data pipeline with orders and inventory data.

In [ ]:
%%sql -r landing_tables
USE SCHEMA SELF_HEALING_PIPELINE.RAW;

-- Orders landing table (simulates incoming e-commerce data)
CREATE OR REPLACE TABLE ORDERS_LANDING (
    ORDER_ID NUMBER(38,0) NOT NULL,
    CUSTOMER_ID NUMBER(38,0),
    ORDER_DATE TIMESTAMP_NTZ,
    PRODUCT_NAME VARCHAR(500),
    QUANTITY NUMBER(10,0),
    UNIT_PRICE NUMBER(12,2),
    TOTAL_AMOUNT NUMBER(12,2),
    STATUS VARCHAR(50),
    REGION VARCHAR(100),
    LOADED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Inventory landing table
CREATE OR REPLACE TABLE INVENTORY_LANDING (
    PRODUCT_ID NUMBER(38,0) NOT NULL,
    PRODUCT_NAME VARCHAR(500),
    WAREHOUSE_LOCATION VARCHAR(200),
    STOCK_LEVEL NUMBER(10,0),
    REORDER_POINT NUMBER(10,0),
    LAST_RESTOCKED TIMESTAMP_NTZ,
    SUPPLIER VARCHAR(300),
    LOADED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

SELECT 'Landing tables created' AS status;

In [ ]:
%%sql -r seed_data
-- Seed with sample data (simulating a healthy load)
INSERT INTO ORDERS_LANDING (ORDER_ID, CUSTOMER_ID, ORDER_DATE, PRODUCT_NAME, QUANTITY, UNIT_PRICE, TOTAL_AMOUNT, STATUS, REGION)
SELECT 
    SEQ4() + 1,
    UNIFORM(1000, 9999, RANDOM()),
    DATEADD('minute', -UNIFORM(0, 43200, RANDOM()), CURRENT_TIMESTAMP()),
    CASE UNIFORM(1,5,RANDOM())
        WHEN 1 THEN 'Wireless Headphones'
        WHEN 2 THEN 'USB-C Hub'
        WHEN 3 THEN 'Mechanical Keyboard'
        WHEN 4 THEN 'Monitor Stand'
        ELSE 'Webcam HD'
    END,
    UNIFORM(1, 20, RANDOM()),
    ROUND(UNIFORM(10, 500, RANDOM())::FLOAT / 10, 2),
    NULL, -- intentionally NULL to test quality checks
    CASE UNIFORM(1,4,RANDOM())
        WHEN 1 THEN 'completed'
        WHEN 2 THEN 'pending'
        WHEN 3 THEN 'shipped'
        ELSE 'cancelled'
    END,
    CASE UNIFORM(1,4,RANDOM())
        WHEN 1 THEN 'North America'
        WHEN 2 THEN 'Europe'
        WHEN 3 THEN 'Asia Pacific'
        ELSE 'Latin America'
    END
FROM TABLE(GENERATOR(ROWCOUNT => 5000));

-- Seed inventory
INSERT INTO INVENTORY_LANDING (PRODUCT_ID, PRODUCT_NAME, WAREHOUSE_LOCATION, STOCK_LEVEL, REORDER_POINT, LAST_RESTOCKED, SUPPLIER)
SELECT
    SEQ4() + 1,
    CASE UNIFORM(1,5,RANDOM())
        WHEN 1 THEN 'Wireless Headphones'
        WHEN 2 THEN 'USB-C Hub'
        WHEN 3 THEN 'Mechanical Keyboard'
        WHEN 4 THEN 'Monitor Stand'
        ELSE 'Webcam HD'
    END,
    CASE UNIFORM(1,3,RANDOM())
        WHEN 1 THEN 'Warehouse-A'
        WHEN 2 THEN 'Warehouse-B'
        ELSE 'Warehouse-C'
    END,
    UNIFORM(0, 500, RANDOM()),
    UNIFORM(10, 50, RANDOM()),
    DATEADD('day', -UNIFORM(1, 90, RANDOM()), CURRENT_DATE()),
    CASE UNIFORM(1,3,RANDOM())
        WHEN 1 THEN 'TechSupply Co'
        WHEN 2 THEN 'GlobalParts Inc'
        ELSE 'FastShip Ltd'
    END
FROM TABLE(GENERATOR(ROWCOUNT => 200));

SELECT 
    (SELECT COUNT(*) FROM ORDERS_LANDING) AS orders_count,
    (SELECT COUNT(*) FROM INVENTORY_LANDING) AS inventory_count;

## 3. Data Quality Monitor
Automated quality checks that run against landing tables and log results to a central table.

In [ ]:
%%sql -r qc_table
USE SCHEMA SELF_HEALING_PIPELINE.MONITORING;

-- Central table to log all quality check results
CREATE OR REPLACE TABLE QUALITY_CHECK_RESULTS (
    CHECK_ID NUMBER AUTOINCREMENT,
    CHECK_TIMESTAMP TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    TABLE_NAME VARCHAR(200),
    CHECK_TYPE VARCHAR(100),
    CHECK_DESCRIPTION VARCHAR(1000),
    PASSED BOOLEAN,
    METRIC_VALUE FLOAT,
    THRESHOLD FLOAT,
    SEVERITY VARCHAR(20), -- LOW, MEDIUM, HIGH, CRITICAL
    DETAILS VARCHAR(5000),
    REMEDIATION_STATUS VARCHAR(50) DEFAULT 'PENDING' -- PENDING, IN_PROGRESS, RESOLVED, ESCALATED
);

SELECT 'Quality check results table created' AS status;

In [ ]:
%%sql -r create_qc_proc
-- Stored procedure: Run all quality checks against ORDERS_LANDING
CREATE OR REPLACE PROCEDURE SELF_HEALING_PIPELINE.MONITORING.RUN_QUALITY_CHECKS()
RETURNS VARCHAR
LANGUAGE SQL
AS
BEGIN
    -- Check 1: NULL rate on TOTAL_AMOUNT (should be < 5%)
    LET null_rate FLOAT;
    SELECT ROUND(COUNT_IF(TOTAL_AMOUNT IS NULL) * 100.0 / COUNT(*), 2)
        INTO :null_rate
        FROM SELF_HEALING_PIPELINE.RAW.ORDERS_LANDING;
    
    LET null_details VARCHAR := 'NULL percentage: ' || :null_rate || '%';
    LET null_passed BOOLEAN := (:null_rate < 5.0);
    LET null_severity VARCHAR := CASE WHEN :null_rate >= 50 THEN 'CRITICAL' WHEN :null_rate >= 20 THEN 'HIGH' WHEN :null_rate >= 5 THEN 'MEDIUM' ELSE 'LOW' END;

    INSERT INTO QUALITY_CHECK_RESULTS (TABLE_NAME, CHECK_TYPE, CHECK_DESCRIPTION, PASSED, METRIC_VALUE, THRESHOLD, SEVERITY, DETAILS)
    VALUES (
        'ORDERS_LANDING', 'NULL_CHECK', 'NULL rate on TOTAL_AMOUNT column',
        :null_passed, :null_rate, 5.0, :null_severity, :null_details
    );

    -- Check 2: Volume check (expect at least 100 rows loaded in last 24h)
    LET recent_count FLOAT;
    SELECT COUNT(*)::FLOAT INTO :recent_count
        FROM SELF_HEALING_PIPELINE.RAW.ORDERS_LANDING
        WHERE LOADED_AT >= DATEADD('hour', -24, CURRENT_TIMESTAMP());
    
    LET vol_details VARCHAR := 'Rows loaded in last 24h: ' || :recent_count;
    LET vol_passed BOOLEAN := (:recent_count >= 100);
    LET vol_severity VARCHAR := CASE WHEN :recent_count = 0 THEN 'CRITICAL' WHEN :recent_count < 50 THEN 'HIGH' WHEN :recent_count < 100 THEN 'MEDIUM' ELSE 'LOW' END;

    INSERT INTO QUALITY_CHECK_RESULTS (TABLE_NAME, CHECK_TYPE, CHECK_DESCRIPTION, PASSED, METRIC_VALUE, THRESHOLD, SEVERITY, DETAILS)
    VALUES (
        'ORDERS_LANDING', 'VOLUME_CHECK', 'Minimum row count in last 24 hours',
        :vol_passed, :recent_count, 100, :vol_severity, :vol_details
    );

    -- Check 3: Duplicate ORDER_ID check
    LET dup_count FLOAT;
    SELECT (COUNT(*) - COUNT(DISTINCT ORDER_ID))::FLOAT INTO :dup_count
        FROM SELF_HEALING_PIPELINE.RAW.ORDERS_LANDING;
    
    LET dup_details VARCHAR := 'Duplicate order IDs found: ' || :dup_count;
    LET dup_passed BOOLEAN := (:dup_count = 0);
    LET dup_severity VARCHAR := CASE WHEN :dup_count > 100 THEN 'CRITICAL' WHEN :dup_count > 10 THEN 'HIGH' WHEN :dup_count > 0 THEN 'MEDIUM' ELSE 'LOW' END;

    INSERT INTO QUALITY_CHECK_RESULTS (TABLE_NAME, CHECK_TYPE, CHECK_DESCRIPTION, PASSED, METRIC_VALUE, THRESHOLD, SEVERITY, DETAILS)
    VALUES (
        'ORDERS_LANDING', 'DUPLICATE_CHECK', 'Duplicate ORDER_ID detection',
        :dup_passed, :dup_count, 0, :dup_severity, :dup_details
    );

    -- Check 4: Schema drift (expected column count)
    LET col_count FLOAT;
    SELECT COUNT(*)::FLOAT INTO :col_count
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA = 'RAW' AND TABLE_NAME = 'ORDERS_LANDING';
    
    LET schema_details VARCHAR := 'Current column count: ' || :col_count || ' (expected: 10)';
    LET schema_passed BOOLEAN := (:col_count = 10);
    LET schema_severity VARCHAR := CASE WHEN :col_count != 10 THEN 'HIGH' ELSE 'LOW' END;

    INSERT INTO QUALITY_CHECK_RESULTS (TABLE_NAME, CHECK_TYPE, CHECK_DESCRIPTION, PASSED, METRIC_VALUE, THRESHOLD, SEVERITY, DETAILS)
    VALUES (
        'ORDERS_LANDING', 'SCHEMA_CHECK', 'Expected column count (10 columns)',
        :schema_passed, :col_count, 10, :schema_severity, :schema_details
    );

    -- Check 5: Value range check (QUANTITY should be 1-1000)
    LET outlier_count FLOAT;
    SELECT COUNT_IF(QUANTITY < 1 OR QUANTITY > 1000)::FLOAT INTO :outlier_count
        FROM SELF_HEALING_PIPELINE.RAW.ORDERS_LANDING;
    
    LET range_details VARCHAR := 'Out-of-range values: ' || :outlier_count;
    LET range_passed BOOLEAN := (:outlier_count = 0);
    LET range_severity VARCHAR := CASE WHEN :outlier_count > 50 THEN 'HIGH' WHEN :outlier_count > 0 THEN 'MEDIUM' ELSE 'LOW' END;

    INSERT INTO QUALITY_CHECK_RESULTS (TABLE_NAME, CHECK_TYPE, CHECK_DESCRIPTION, PASSED, METRIC_VALUE, THRESHOLD, SEVERITY, DETAILS)
    VALUES (
        'ORDERS_LANDING', 'RANGE_CHECK', 'QUANTITY values within expected range (1-1000)',
        :range_passed, :outlier_count, 0, :range_severity, :range_details
    );

    RETURN 'Quality checks completed. 5 checks executed.';
END;

In [ ]:
%%sql -r qc_results
-- Execute quality checks
CALL SELF_HEALING_PIPELINE.MONITORING.RUN_QUALITY_CHECKS();

-- View results
SELECT CHECK_ID, TABLE_NAME, CHECK_TYPE, PASSED, METRIC_VALUE, THRESHOLD, SEVERITY, DETAILS
FROM SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS
ORDER BY CHECK_TIMESTAMP DESC;

## 4. Anomaly Detection Stream
A stream captures new quality check failures in real-time, feeding them into the AI classifier.

In [ ]:
%%sql -r stream_data
-- Stream to detect new quality failures
CREATE OR REPLACE STREAM SELF_HEALING_PIPELINE.MONITORING.QUALITY_FAILURES_STREAM
    ON TABLE SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS
    APPEND_ONLY = TRUE
    SHOW_INITIAL_ROWS = TRUE;

-- View current stream contents (failed checks only)
SELECT * FROM SELF_HEALING_PIPELINE.MONITORING.QUALITY_FAILURES_STREAM
WHERE PASSED = FALSE
ORDER BY CHECK_TIMESTAMP DESC;

## 5. AI-Powered Anomaly Classifier
Uses Snowflake Cortex `COMPLETE` to analyze each failure, determine root cause, and recommend remediation.

In [ ]:
%%sql -r ai_tables
USE SCHEMA SELF_HEALING_PIPELINE.OPERATIONS;

-- Table to store AI classifications
CREATE OR REPLACE TABLE ANOMALY_CLASSIFICATIONS (
    CLASSIFICATION_ID NUMBER AUTOINCREMENT,
    CHECK_ID NUMBER,
    CLASSIFIED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    AI_SEVERITY VARCHAR(20),
    ROOT_CAUSE VARCHAR(2000),
    RECOMMENDED_ACTION VARCHAR(2000),
    AUTO_REMEDIABLE BOOLEAN,
    CONFIDENCE_SCORE FLOAT,
    RAW_AI_RESPONSE VARCHAR(10000)
);

-- Audit log for all remediation actions
CREATE OR REPLACE TABLE REMEDIATION_AUDIT_LOG (
    LOG_ID NUMBER AUTOINCREMENT,
    CHECK_ID NUMBER,
    ACTION_TAKEN VARCHAR(1000),
    ACTION_TIMESTAMP TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    SUCCESS BOOLEAN,
    DETAILS VARCHAR(5000),
    ACTOR VARCHAR(100) DEFAULT 'SYSTEM' -- SYSTEM or HUMAN
);

SELECT 'AI classification and audit tables created' AS status;

In [ ]:
%%sql -r classify_proc
-- Procedure: Classify anomalies using rule-based engine
-- (On paid accounts, replace with SNOWFLAKE.CORTEX.COMPLETE for AI classification)
CREATE OR REPLACE PROCEDURE SELF_HEALING_PIPELINE.OPERATIONS.CLASSIFY_ANOMALIES()
RETURNS VARCHAR
LANGUAGE SQL
AS
BEGIN
    LET processed NUMBER := 0;

    INSERT INTO SELF_HEALING_PIPELINE.OPERATIONS.ANOMALY_CLASSIFICATIONS 
        (CHECK_ID, AI_SEVERITY, ROOT_CAUSE, RECOMMENDED_ACTION, AUTO_REMEDIABLE, CONFIDENCE_SCORE, RAW_AI_RESPONSE)
    SELECT 
        q.CHECK_ID,
        q.SEVERITY,
        CASE q.CHECK_TYPE
            WHEN 'NULL_CHECK' THEN 'Upstream ETL failed to populate computed columns. Source system may have changed schema or calculation logic.'
            WHEN 'VOLUME_CHECK' THEN 'Data source may be offline, ingestion job failed, or upstream system experienced an outage.'
            WHEN 'DUPLICATE_CHECK' THEN 'Ingestion job ran multiple times without idempotency or source system sent duplicate records.'
            WHEN 'SCHEMA_CHECK' THEN 'Schema drift detected. An ALTER TABLE was executed outside the pipeline or a migration ran unexpectedly.'
            WHEN 'RANGE_CHECK' THEN 'Data validation at source is missing. Outlier values indicate corrupt records or unit mismatch.'
            ELSE 'Unknown check type. Manual investigation required.'
        END,
        CASE q.CHECK_TYPE
            WHEN 'NULL_CHECK' THEN 'Recompute NULL values using available columns (QUANTITY * UNIT_PRICE for TOTAL_AMOUNT).'
            WHEN 'VOLUME_CHECK' THEN 'Check ingestion job logs. Verify source connectivity. Re-trigger load if source is available.'
            WHEN 'DUPLICATE_CHECK' THEN 'Deduplicate by keeping most recent record per primary key.'
            WHEN 'SCHEMA_CHECK' THEN 'Compare current schema against expected. Revert unauthorized changes.'
            WHEN 'RANGE_CHECK' THEN 'Quarantine out-of-range records to a holding table.'
            ELSE 'Escalate to data engineering team for manual review.'
        END,
        CASE q.CHECK_TYPE
            WHEN 'NULL_CHECK' THEN TRUE
            WHEN 'DUPLICATE_CHECK' THEN TRUE
            WHEN 'RANGE_CHECK' THEN TRUE
            ELSE FALSE
        END,
        CASE 
            WHEN q.METRIC_VALUE > q.THRESHOLD * 10 THEN 0.95
            WHEN q.METRIC_VALUE > q.THRESHOLD * 5 THEN 0.85
            ELSE 0.70
        END,
        'Rule-based classification engine'
    FROM SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS q
    WHERE q.PASSED = FALSE 
      AND q.REMEDIATION_STATUS = 'PENDING'
      AND q.CHECK_ID NOT IN (SELECT CHECK_ID FROM SELF_HEALING_PIPELINE.OPERATIONS.ANOMALY_CLASSIFICATIONS);

    SELECT COUNT(*) INTO :processed
    FROM SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS
    WHERE PASSED = FALSE AND REMEDIATION_STATUS = 'PENDING'
      AND CHECK_ID IN (SELECT CHECK_ID FROM SELF_HEALING_PIPELINE.OPERATIONS.ANOMALY_CLASSIFICATIONS);

    UPDATE SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS
    SET REMEDIATION_STATUS = 'IN_PROGRESS'
    WHERE PASSED = FALSE 
      AND REMEDIATION_STATUS = 'PENDING'
      AND CHECK_ID IN (SELECT CHECK_ID FROM SELF_HEALING_PIPELINE.OPERATIONS.ANOMALY_CLASSIFICATIONS);

    RETURN 'Classified ' || :processed || ' anomalies.';
END;

In [ ]:
%%sql -r classifications
-- Run the AI classifier on current failures
CALL SELF_HEALING_PIPELINE.OPERATIONS.CLASSIFY_ANOMALIES();

-- View AI classifications
SELECT c.CHECK_ID, c.AI_SEVERITY, c.ROOT_CAUSE, c.RECOMMENDED_ACTION, 
       c.AUTO_REMEDIABLE, c.CONFIDENCE_SCORE, c.CLASSIFIED_AT
FROM SELF_HEALING_PIPELINE.OPERATIONS.ANOMALY_CLASSIFICATIONS c
ORDER BY c.CLASSIFIED_AT DESC;

## 6. Auto-Remediation Engine
Attempts to fix issues automatically based on AI recommendations. Each action type has a specific handler.

In [ ]:
%%sql -r remediate_proc
-- Procedure: Attempt automated remediation based on classification
CREATE OR REPLACE PROCEDURE SELF_HEALING_PIPELINE.OPERATIONS.AUTO_REMEDIATE()
RETURNS VARCHAR
LANGUAGE SQL
AS
BEGIN
    LET remediated NUMBER := 0;
    LET escalated NUMBER := 0;

    LET cur CURSOR FOR
        SELECT c.CLASSIFICATION_ID, c.CHECK_ID, c.ROOT_CAUSE, c.RECOMMENDED_ACTION, c.AUTO_REMEDIABLE,
               q.TABLE_NAME, q.CHECK_TYPE, q.METRIC_VALUE
        FROM SELF_HEALING_PIPELINE.OPERATIONS.ANOMALY_CLASSIFICATIONS c
        JOIN SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS q ON c.CHECK_ID = q.CHECK_ID
        WHERE q.REMEDIATION_STATUS = 'IN_PROGRESS'
          AND c.AUTO_REMEDIABLE = TRUE;

    FOR record IN cur DO
        LET success BOOLEAN := FALSE;
        LET action_taken VARCHAR := '';
        LET current_check_id NUMBER := record.CHECK_ID;
        LET current_check_type VARCHAR := record.CHECK_TYPE;
        LET current_table VARCHAR := record.TABLE_NAME;
        LET current_metric VARCHAR := record.METRIC_VALUE::VARCHAR;

        -- Remediation: Fix NULL values
        IF (:current_check_type = 'NULL_CHECK' AND :current_table = 'ORDERS_LANDING') THEN
            UPDATE SELF_HEALING_PIPELINE.RAW.ORDERS_LANDING
            SET TOTAL_AMOUNT = QUANTITY * UNIT_PRICE
            WHERE TOTAL_AMOUNT IS NULL AND QUANTITY IS NOT NULL AND UNIT_PRICE IS NOT NULL;
            action_taken := 'Computed TOTAL_AMOUNT = QUANTITY * UNIT_PRICE for NULL rows';
            success := TRUE;
        END IF;

        -- Remediation: Remove duplicates
        IF (:current_check_type = 'DUPLICATE_CHECK') THEN
            DELETE FROM SELF_HEALING_PIPELINE.RAW.ORDERS_LANDING
            WHERE ORDER_ID IN (
                SELECT ORDER_ID FROM (
                    SELECT ORDER_ID, ROW_NUMBER() OVER (PARTITION BY ORDER_ID ORDER BY LOADED_AT DESC) AS rn
                    FROM SELF_HEALING_PIPELINE.RAW.ORDERS_LANDING
                ) WHERE rn > 1
            );
            action_taken := 'Removed duplicate ORDER_IDs, kept most recent';
            success := TRUE;
        END IF;

        -- Remediation: Quarantine out-of-range
        IF (:current_check_type = 'RANGE_CHECK') THEN
            CREATE TABLE IF NOT EXISTS SELF_HEALING_PIPELINE.RAW.ORDERS_QUARANTINE LIKE SELF_HEALING_PIPELINE.RAW.ORDERS_LANDING;
            INSERT INTO SELF_HEALING_PIPELINE.RAW.ORDERS_QUARANTINE
            SELECT * FROM SELF_HEALING_PIPELINE.RAW.ORDERS_LANDING
            WHERE QUANTITY < 1 OR QUANTITY > 1000;
            DELETE FROM SELF_HEALING_PIPELINE.RAW.ORDERS_LANDING
            WHERE QUANTITY < 1 OR QUANTITY > 1000;
            action_taken := 'Quarantined rows with out-of-range QUANTITY values';
            success := TRUE;
        END IF;

        -- Log the result
        IF (:action_taken != '') THEN
            LET log_detail VARCHAR := 'Metric value: ' || :current_metric;
            INSERT INTO SELF_HEALING_PIPELINE.OPERATIONS.REMEDIATION_AUDIT_LOG (CHECK_ID, ACTION_TAKEN, SUCCESS, DETAILS)
            VALUES (:current_check_id, :action_taken, :success, :log_detail);

            UPDATE SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS
            SET REMEDIATION_STATUS = CASE WHEN :success THEN 'RESOLVED' ELSE 'ESCALATED' END
            WHERE CHECK_ID = :current_check_id;
            remediated := remediated + 1;
        ELSE
            UPDATE SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS
            SET REMEDIATION_STATUS = 'ESCALATED'
            WHERE CHECK_ID = :current_check_id;

            LET esc_detail VARCHAR := record.RECOMMENDED_ACTION;
            INSERT INTO SELF_HEALING_PIPELINE.OPERATIONS.REMEDIATION_AUDIT_LOG (CHECK_ID, ACTION_TAKEN, SUCCESS, DETAILS)
            VALUES (:current_check_id, 'ESCALATED - No auto-fix available', FALSE, :esc_detail);
            escalated := escalated + 1;
        END IF;
    END FOR;

    -- Escalate non-auto-remediable items
    UPDATE SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS q
    SET REMEDIATION_STATUS = 'ESCALATED'
    WHERE q.REMEDIATION_STATUS = 'IN_PROGRESS'
      AND q.CHECK_ID IN (
          SELECT CHECK_ID FROM SELF_HEALING_PIPELINE.OPERATIONS.ANOMALY_CLASSIFICATIONS
          WHERE AUTO_REMEDIABLE = FALSE
      );

    RETURN 'Remediation complete. Fixed: ' || :remediated || ', Escalated: ' || :escalated;
END;

In [ ]:
%%sql -r remediation_log
-- Execute auto-remediation
CALL SELF_HEALING_PIPELINE.OPERATIONS.AUTO_REMEDIATE();

-- View audit log
SELECT * FROM SELF_HEALING_PIPELINE.OPERATIONS.REMEDIATION_AUDIT_LOG
ORDER BY ACTION_TIMESTAMP DESC;

## 7. Scheduled Automation (Tasks)
Chain the entire pipeline into a scheduled task graph that runs autonomously.

In [ ]:
%%sql -r tasks_created
-- Master orchestrator task: runs quality checks every 5 minutes
CREATE OR REPLACE TASK SELF_HEALING_PIPELINE.OPERATIONS.TASK_RUN_QUALITY_CHECKS
    WAREHOUSE = COMPUTE_WH
    SCHEDULE = '5 MINUTE'
AS
    CALL SELF_HEALING_PIPELINE.MONITORING.RUN_QUALITY_CHECKS();

-- Child task: classifies anomalies (runs after quality checks)
CREATE OR REPLACE TASK SELF_HEALING_PIPELINE.OPERATIONS.TASK_CLASSIFY_ANOMALIES
    WAREHOUSE = COMPUTE_WH
    AFTER SELF_HEALING_PIPELINE.OPERATIONS.TASK_RUN_QUALITY_CHECKS
AS
    CALL SELF_HEALING_PIPELINE.OPERATIONS.CLASSIFY_ANOMALIES();

-- Child task: auto-remediation (runs after classification)
CREATE OR REPLACE TASK SELF_HEALING_PIPELINE.OPERATIONS.TASK_AUTO_REMEDIATE
    WAREHOUSE = COMPUTE_WH
    AFTER SELF_HEALING_PIPELINE.OPERATIONS.TASK_CLASSIFY_ANOMALIES
AS
    CALL SELF_HEALING_PIPELINE.OPERATIONS.AUTO_REMEDIATE();

SELECT 'Task graph created (currently suspended). Resume to activate.' AS status;

In [ ]:
%%sql -r task_status
-- Resume the root task to activate the entire pipeline
-- (Uncomment the line below when ready to run autonomously)
-- ALTER TASK SELF_HEALING_PIPELINE.OPERATIONS.TASK_RUN_QUALITY_CHECKS RESUME;

-- View the task graph
SHOW TASKS IN SCHEMA SELF_HEALING_PIPELINE.OPERATIONS;

## 8. Escalation & Alerting
For issues the system cannot auto-fix, escalate via email notification.

In [ ]:
%%sql -r escalated
-- Create a view of escalated issues for alerting
CREATE OR REPLACE VIEW SELF_HEALING_PIPELINE.OPERATIONS.ESCALATED_ISSUES AS
SELECT 
    q.CHECK_ID,
    q.CHECK_TIMESTAMP,
    q.TABLE_NAME,
    q.CHECK_TYPE,
    q.CHECK_DESCRIPTION,
    q.SEVERITY,
    q.DETAILS,
    c.ROOT_CAUSE,
    c.RECOMMENDED_ACTION,
    c.CONFIDENCE_SCORE
FROM SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS q
LEFT JOIN SELF_HEALING_PIPELINE.OPERATIONS.ANOMALY_CLASSIFICATIONS c ON q.CHECK_ID = c.CHECK_ID
WHERE q.REMEDIATION_STATUS = 'ESCALATED'
ORDER BY q.CHECK_TIMESTAMP DESC;

-- View escalated issues
SELECT * FROM SELF_HEALING_PIPELINE.OPERATIONS.ESCALATED_ISSUES;

In [ ]:
%%sql -r alert_status
-- Email notification integration (template - configure with your email service)
-- Uncomment and configure when ready to enable alerts:

/*
CREATE OR REPLACE NOTIFICATION INTEGRATION PIPELINE_ALERTS
    TYPE = EMAIL
    ENABLED = TRUE
    ALLOWED_RECIPIENTS = ('data-team@yourcompany.com');

-- Procedure to send escalation alerts
CREATE OR REPLACE PROCEDURE SELF_HEALING_PIPELINE.OPERATIONS.SEND_ESCALATION_ALERT()
RETURNS VARCHAR
LANGUAGE SQL
AS
BEGIN
    LET issue_count NUMBER;
    SELECT COUNT(*) INTO :issue_count FROM SELF_HEALING_PIPELINE.OPERATIONS.ESCALATED_ISSUES;
    
    IF (:issue_count > 0) THEN
        CALL SYSTEM$SEND_EMAIL(
            'PIPELINE_ALERTS',
            'data-team@yourcompany.com',
            'ALERT: Self-Healing Pipeline - ' || :issue_count || ' Escalated Issues',
            'The self-healing pipeline has ' || :issue_count || ' issues requiring manual review. Check the ESCALATED_ISSUES view for details.'
        );
    END IF;
    RETURN 'Alert sent for ' || :issue_count || ' issues';
END;
*/

SELECT 'Notification integration template ready (configure email to activate)' AS status;

## 9. Pipeline Health Dashboard
Real-time visibility into pipeline health, remediation success rate, and anomaly trends.

In [ ]:
%%sql -r health_summary
-- Pipeline health summary
SELECT 
    COUNT(*) AS total_checks,
    COUNT_IF(PASSED = TRUE) AS passed,
    COUNT_IF(PASSED = FALSE) AS failed,
    ROUND(COUNT_IF(PASSED = TRUE) * 100.0 / NULLIF(COUNT(*), 0), 1) AS health_score_pct,
    COUNT_IF(REMEDIATION_STATUS = 'RESOLVED') AS auto_resolved,
    COUNT_IF(REMEDIATION_STATUS = 'ESCALATED') AS escalated,
    COUNT_IF(REMEDIATION_STATUS = 'PENDING') AS pending
FROM SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS;

In [ ]:
import pandas as pd

# Get pipeline health data
health_pdf = health_summary.to_pandas() if not isinstance(health_summary, pd.DataFrame) else health_summary.copy()

print("=" * 60)
print("       SELF-HEALING PIPELINE - HEALTH REPORT")
print("=" * 60)

if not health_pdf.empty:
    row = health_pdf.iloc[0]
    score = row['HEALTH_SCORE_PCT']
    
    # Health indicator
    if score >= 90:
        status_icon = "HEALTHY"
    elif score >= 70:
        status_icon = "WARNING"
    else:
        status_icon = "CRITICAL"
    
    print(f"\n  Status: {status_icon}")
    print(f"  Health Score: {score}%")
    print(f"\n  Total Checks Run: {int(row['TOTAL_CHECKS'])}")
    print(f"  Passed: {int(row['PASSED'])}")
    print(f"  Failed: {int(row['FAILED'])}")
    print(f"\n  Auto-Resolved: {int(row['AUTO_RESOLVED'])}")
    print(f"  Escalated: {int(row['ESCALATED'])}")
    print(f"  Pending: {int(row['PENDING'])}")
    
    # Remediation effectiveness
    total_issues = int(row['AUTO_RESOLVED']) + int(row['ESCALATED'])
    if total_issues > 0:
        effectiveness = int(row['AUTO_RESOLVED']) * 100.0 / total_issues
        print(f"\n  Auto-Remediation Rate: {effectiveness:.1f}%")

print("\n" + "=" * 60)

In [ ]:
%%sql -r severity_dist
-- Severity distribution of anomalies
SELECT 
    SEVERITY,
    COUNT(*) AS count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS percentage
FROM SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS
WHERE PASSED = FALSE
GROUP BY SEVERITY
ORDER BY 
    CASE SEVERITY 
        WHEN 'CRITICAL' THEN 1 
        WHEN 'HIGH' THEN 2 
        WHEN 'MEDIUM' THEN 3 
        ELSE 4 
    END;

In [ ]:
%%sql -r remediation_timeline
-- Remediation history timeline
SELECT 
    DATE_TRUNC('hour', ACTION_TIMESTAMP) AS hour,
    COUNT_IF(SUCCESS = TRUE) AS successful_fixes,
    COUNT_IF(SUCCESS = FALSE) AS failed_fixes,
    COUNT(*) AS total_actions
FROM SELF_HEALING_PIPELINE.OPERATIONS.REMEDIATION_AUDIT_LOG
GROUP BY 1
ORDER BY 1 DESC
LIMIT 24;

## 10. Simulate a Data Incident
Inject bad data to demonstrate the self-healing pipeline in action.

In [ ]:
%%sql -r inject_bad_data
-- Inject anomalies to test the pipeline
-- 1. Add rows with NULL critical fields
INSERT INTO SELF_HEALING_PIPELINE.RAW.ORDERS_LANDING 
    (ORDER_ID, CUSTOMER_ID, ORDER_DATE, PRODUCT_NAME, QUANTITY, UNIT_PRICE, TOTAL_AMOUNT, STATUS, REGION)
SELECT 
    SEQ4() + 10000,
    NULL, -- NULL customer
    CURRENT_TIMESTAMP(),
    'Test Product',
    UNIFORM(1, 10, RANDOM()),
    UNIFORM(10, 100, RANDOM()),
    NULL, -- NULL total
    'pending',
    'Unknown'
FROM TABLE(GENERATOR(ROWCOUNT => 500));

-- 2. Add duplicate ORDER_IDs
INSERT INTO SELF_HEALING_PIPELINE.RAW.ORDERS_LANDING 
    (ORDER_ID, CUSTOMER_ID, ORDER_DATE, PRODUCT_NAME, QUANTITY, UNIT_PRICE, TOTAL_AMOUNT, STATUS, REGION)
SELECT 
    1, -- duplicate ID
    9999,
    CURRENT_TIMESTAMP(),
    'Duplicate Test',
    5,
    25.00,
    125.00,
    'completed',
    'North America'
FROM TABLE(GENERATOR(ROWCOUNT => 15));

SELECT 'Bad data injected! Run the pipeline to see self-healing in action.' AS status;

In [ ]:
%%sql -r full_cycle
-- Run the full self-healing cycle manually
CALL SELF_HEALING_PIPELINE.MONITORING.RUN_QUALITY_CHECKS();
CALL SELF_HEALING_PIPELINE.OPERATIONS.CLASSIFY_ANOMALIES();
CALL SELF_HEALING_PIPELINE.OPERATIONS.AUTO_REMEDIATE();

-- See the results
SELECT 
    q.CHECK_TYPE, q.SEVERITY, q.PASSED, q.REMEDIATION_STATUS,
    c.ROOT_CAUSE, c.RECOMMENDED_ACTION, c.AUTO_REMEDIABLE,
    r.ACTION_TAKEN, r.SUCCESS
FROM SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS q
LEFT JOIN SELF_HEALING_PIPELINE.OPERATIONS.ANOMALY_CLASSIFICATIONS c ON q.CHECK_ID = c.CHECK_ID
LEFT JOIN SELF_HEALING_PIPELINE.OPERATIONS.REMEDIATION_AUDIT_LOG r ON q.CHECK_ID = r.CHECK_ID
WHERE q.PASSED = FALSE
ORDER BY q.CHECK_TIMESTAMP DESC
LIMIT 20;

## 11. Visual Dashboard
Interactive charts showing pipeline health, anomaly severity, and remediation performance.

In [ ]:
%%sql -r dashboard_data
-- Dashboard data: all checks with classifications and remediations
SELECT 
    q.CHECK_ID, q.CHECK_TIMESTAMP, q.TABLE_NAME, q.CHECK_TYPE,
    q.PASSED, q.METRIC_VALUE, q.THRESHOLD, q.SEVERITY, q.REMEDIATION_STATUS,
    c.ROOT_CAUSE, c.RECOMMENDED_ACTION, c.AUTO_REMEDIABLE, c.CONFIDENCE_SCORE,
    r.ACTION_TAKEN, r.SUCCESS AS REMEDIATION_SUCCESS
FROM SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS q
LEFT JOIN SELF_HEALING_PIPELINE.OPERATIONS.ANOMALY_CLASSIFICATIONS c ON q.CHECK_ID = c.CHECK_ID
LEFT JOIN SELF_HEALING_PIPELINE.OPERATIONS.REMEDIATION_AUDIT_LOG r ON q.CHECK_ID = r.CHECK_ID
ORDER BY q.CHECK_TIMESTAMP DESC;

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Load data
df = dashboard_data.to_pandas() if not isinstance(dashboard_data, pd.DataFrame) else dashboard_data.copy()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('SELF-HEALING PIPELINE DASHBOARD', fontsize=16, fontweight='bold', y=0.98)

# --- Panel 1: Health Score Gauge ---
ax1 = axes[0, 0]
total = len(df)
passed = df['PASSED'].sum()
failed = total - passed
health_pct = (passed / total * 100) if total > 0 else 0

colors = ['#2ecc71' if health_pct >= 80 else '#f39c12' if health_pct >= 60 else '#e74c3c', '#ecf0f1']
ax1.pie([health_pct, 100 - health_pct], colors=colors, startangle=90,
        wedgeprops={'width': 0.4, 'edgecolor': 'white'})
ax1.text(0, 0, f'{health_pct:.0f}%', ha='center', va='center', fontsize=28, fontweight='bold')
ax1.set_title('Health Score', fontsize=12, fontweight='bold')

# --- Panel 2: Severity Distribution ---
ax2 = axes[0, 1]
failed_df = df[df['PASSED'] == False]
if not failed_df.empty:
    sev_counts = failed_df['SEVERITY'].value_counts()
    sev_colors = {'CRITICAL': '#e74c3c', 'HIGH': '#e67e22', 'MEDIUM': '#f1c40f', 'LOW': '#2ecc71'}
    colors_list = [sev_colors.get(s, '#95a5a6') for s in sev_counts.index]
    bars = ax2.bar(sev_counts.index, sev_counts.values, color=colors_list, edgecolor='white')
    for bar, val in zip(bars, sev_counts.values):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                str(val), ha='center', va='bottom', fontweight='bold')
ax2.set_title('Failures by Severity', fontsize=12, fontweight='bold')
ax2.set_ylabel('Count')

# --- Panel 3: Remediation Status ---
ax3 = axes[1, 0]
status_counts = df['REMEDIATION_STATUS'].value_counts()
status_colors = {'RESOLVED': '#2ecc71', 'ESCALATED': '#e74c3c', 'PENDING': '#f1c40f', 'IN_PROGRESS': '#3498db'}
colors_list = [status_colors.get(s, '#95a5a6') for s in status_counts.index]
wedges, texts, autotexts = ax3.pie(status_counts.values, labels=status_counts.index,
                                    colors=colors_list, autopct='%1.0f%%',
                                    textprops={'fontsize': 9})
ax3.set_title('Remediation Status', fontsize=12, fontweight='bold')

# --- Panel 4: Check Type Results ---
ax4 = axes[1, 1]
check_summary = df.groupby('CHECK_TYPE')['PASSED'].agg(['sum', 'count']).reset_index()
check_summary.columns = ['CHECK_TYPE', 'PASSED', 'TOTAL']
check_summary['FAILED'] = check_summary['TOTAL'] - check_summary['PASSED']

x = range(len(check_summary))
width = 0.35
ax4.bar([i - width/2 for i in x], check_summary['PASSED'], width, label='Passed', color='#2ecc71')
ax4.bar([i + width/2 for i in x], check_summary['FAILED'], width, label='Failed', color='#e74c3c')
ax4.set_xticks(list(x))
ax4.set_xticklabels(check_summary['CHECK_TYPE'], rotation=30, ha='right', fontsize=8)
ax4.set_title('Results by Check Type', fontsize=12, fontweight='bold')
ax4.set_ylabel('Count')
ax4.legend()

plt.tight_layout()
plt.subplots_adjust(top=0.93)
plt.show()

# Print summary stats
print(f"\n{'='*50}")
print(f"  Total Checks: {total} | Passed: {passed} | Failed: {failed}")
resolved = (df['REMEDIATION_STATUS'] == 'RESOLVED').sum()
print(f"  Auto-Resolved: {resolved} | Remediation Rate: {resolved}/{failed if failed > 0 else 1} = {resolved/max(failed,1)*100:.0f}%")
print(f"{'='*50}")

---
# ADVANCED ENHANCEMENTS
---

## 12. Anomaly Pattern Learning
Track which issues repeat, identify patterns, and predict future failures before they happen.

In [ ]:
%%sql -r patterns
USE SCHEMA SELF_HEALING_PIPELINE.MONITORING;

-- Pattern history table: tracks recurring anomalies
CREATE OR REPLACE TABLE ANOMALY_PATTERNS (
    PATTERN_ID NUMBER AUTOINCREMENT,
    TABLE_NAME VARCHAR(200),
    CHECK_TYPE VARCHAR(100),
    OCCURRENCE_COUNT NUMBER DEFAULT 1,
    FIRST_SEEN TIMESTAMP_NTZ,
    LAST_SEEN TIMESTAMP_NTZ,
    AVG_INTERVAL_HOURS FLOAT,
    PREDICTED_NEXT_OCCURRENCE TIMESTAMP_NTZ,
    PATTERN_STATUS VARCHAR(50) DEFAULT 'ACTIVE', -- ACTIVE, RESOLVED, CHRONIC
    TREND VARCHAR(20) DEFAULT 'STABLE', -- IMPROVING, STABLE, WORSENING
    LAST_UPDATED TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Procedure: Analyze patterns from historical check results
CREATE OR REPLACE PROCEDURE SELF_HEALING_PIPELINE.MONITORING.ANALYZE_PATTERNS()
RETURNS VARCHAR
LANGUAGE SQL
AS
BEGIN
    -- Merge pattern data from recent failures
    MERGE INTO ANOMALY_PATTERNS p
    USING (
        SELECT 
            TABLE_NAME,
            CHECK_TYPE,
            COUNT(*) AS total_failures,
            MIN(CHECK_TIMESTAMP) AS first_failure,
            MAX(CHECK_TIMESTAMP) AS last_failure,
            CASE WHEN COUNT(*) > 1 
                THEN TIMESTAMPDIFF('hour', MIN(CHECK_TIMESTAMP), MAX(CHECK_TIMESTAMP)) / (COUNT(*) - 1)
                ELSE NULL 
            END AS avg_interval
        FROM QUALITY_CHECK_RESULTS
        WHERE PASSED = FALSE
        GROUP BY TABLE_NAME, CHECK_TYPE
    ) src
    ON p.TABLE_NAME = src.TABLE_NAME AND p.CHECK_TYPE = src.CHECK_TYPE
    WHEN MATCHED THEN UPDATE SET
        p.OCCURRENCE_COUNT = src.total_failures,
        p.FIRST_SEEN = src.first_failure,
        p.LAST_SEEN = src.last_failure,
        p.AVG_INTERVAL_HOURS = src.avg_interval,
        p.PREDICTED_NEXT_OCCURRENCE = CASE 
            WHEN src.avg_interval IS NOT NULL 
            THEN DATEADD('hour', src.avg_interval::INT, src.last_failure)
            ELSE NULL END,
        p.TREND = CASE
            WHEN src.total_failures > p.OCCURRENCE_COUNT * 1.5 THEN 'WORSENING'
            WHEN src.total_failures < p.OCCURRENCE_COUNT * 0.8 THEN 'IMPROVING'
            ELSE 'STABLE' END,
        p.PATTERN_STATUS = CASE
            WHEN src.total_failures > 20 THEN 'CHRONIC'
            WHEN TIMESTAMPDIFF('hour', src.last_failure, CURRENT_TIMESTAMP()) > 48 THEN 'RESOLVED'
            ELSE 'ACTIVE' END,
        p.LAST_UPDATED = CURRENT_TIMESTAMP()
    WHEN NOT MATCHED THEN INSERT (TABLE_NAME, CHECK_TYPE, OCCURRENCE_COUNT, FIRST_SEEN, LAST_SEEN, AVG_INTERVAL_HOURS, PREDICTED_NEXT_OCCURRENCE)
    VALUES (src.TABLE_NAME, src.CHECK_TYPE, src.total_failures, src.first_failure, src.last_failure, src.avg_interval,
            CASE WHEN src.avg_interval IS NOT NULL THEN DATEADD('hour', src.avg_interval::INT, src.last_failure) ELSE NULL END);

    RETURN 'Pattern analysis complete.';
END;

CALL SELF_HEALING_PIPELINE.MONITORING.ANALYZE_PATTERNS();

-- View detected patterns with predictions
SELECT 
    TABLE_NAME, CHECK_TYPE, OCCURRENCE_COUNT, TREND, PATTERN_STATUS,
    AVG_INTERVAL_HOURS,
    PREDICTED_NEXT_OCCURRENCE,
    CASE 
        WHEN PREDICTED_NEXT_OCCURRENCE < CURRENT_TIMESTAMP() THEN 'OVERDUE - likely failing now'
        WHEN PREDICTED_NEXT_OCCURRENCE < DATEADD('hour', 6, CURRENT_TIMESTAMP()) THEN 'WARNING - expected within 6 hours'
        ELSE 'OK - not expected soon'
    END AS PREDICTION_ALERT
FROM ANOMALY_PATTERNS
ORDER BY OCCURRENCE_COUNT DESC;

## 13. Cross-Table Dependency Awareness
If an upstream table fails quality checks, automatically pause downstream consumers to prevent bad data from propagating.

In [ ]:
%%sql -r dependencies
USE SCHEMA SELF_HEALING_PIPELINE.OPERATIONS;

-- Table dependency graph
CREATE OR REPLACE TABLE TABLE_DEPENDENCIES (
    DEPENDENCY_ID NUMBER AUTOINCREMENT,
    UPSTREAM_TABLE VARCHAR(500),
    DOWNSTREAM_TABLE VARCHAR(500),
    DEPENDENCY_TYPE VARCHAR(50), -- HARD (must pause), SOFT (warn only)
    AUTO_PAUSE_ON_FAILURE BOOLEAN DEFAULT TRUE,
    CREATED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Downstream status tracking
CREATE OR REPLACE TABLE DOWNSTREAM_PAUSE_LOG (
    LOG_ID NUMBER AUTOINCREMENT,
    UPSTREAM_TABLE VARCHAR(500),
    DOWNSTREAM_TABLE VARCHAR(500),
    ACTION VARCHAR(50), -- PAUSED, RESUMED
    REASON VARCHAR(1000),
    ACTION_TIMESTAMP TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Define sample dependencies
INSERT INTO TABLE_DEPENDENCIES (UPSTREAM_TABLE, DOWNSTREAM_TABLE, DEPENDENCY_TYPE, AUTO_PAUSE_ON_FAILURE)
VALUES 
    ('RAW.ORDERS_LANDING', 'ANALYTICS.ORDERS_FACT', 'HARD', TRUE),
    ('RAW.ORDERS_LANDING', 'ANALYTICS.REVENUE_DAILY', 'HARD', TRUE),
    ('RAW.ORDERS_LANDING', 'ML.DEMAND_FORECAST_INPUT', 'SOFT', FALSE),
    ('RAW.INVENTORY_LANDING', 'ANALYTICS.INVENTORY_SNAPSHOT', 'HARD', TRUE),
    ('RAW.INVENTORY_LANDING', 'ANALYTICS.REORDER_ALERTS', 'HARD', TRUE);

-- Procedure: Check dependencies and pause downstream if upstream is unhealthy
CREATE OR REPLACE PROCEDURE SELF_HEALING_PIPELINE.OPERATIONS.CHECK_DEPENDENCIES()
RETURNS VARCHAR
LANGUAGE SQL
AS
BEGIN
    LET paused_count NUMBER := 0;
    LET resumed_count NUMBER := 0;

    -- Find upstream tables with recent CRITICAL/HIGH failures
    LET cur CURSOR FOR
        SELECT DISTINCT
            d.UPSTREAM_TABLE,
            d.DOWNSTREAM_TABLE,
            d.DEPENDENCY_TYPE,
            d.AUTO_PAUSE_ON_FAILURE,
            q.SEVERITY,
            q.REMEDIATION_STATUS
        FROM TABLE_DEPENDENCIES d
        JOIN SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS q
            ON q.TABLE_NAME = SPLIT_PART(d.UPSTREAM_TABLE, '.', 2)
        WHERE q.PASSED = FALSE
          AND q.SEVERITY IN ('CRITICAL', 'HIGH')
          AND q.REMEDIATION_STATUS NOT IN ('RESOLVED')
          AND q.CHECK_TIMESTAMP >= DATEADD('hour', -1, CURRENT_TIMESTAMP())
          AND d.AUTO_PAUSE_ON_FAILURE = TRUE;

    FOR rec IN cur DO
        LET up_tbl VARCHAR := rec.UPSTREAM_TABLE;
        LET down_tbl VARCHAR := rec.DOWNSTREAM_TABLE;
        LET sev VARCHAR := rec.SEVERITY;
        LET reason VARCHAR := 'Upstream ' || :up_tbl || ' has ' || :sev || ' failure. Auto-pausing to prevent bad data propagation.';

        INSERT INTO DOWNSTREAM_PAUSE_LOG (UPSTREAM_TABLE, DOWNSTREAM_TABLE, ACTION, REASON)
        VALUES (:up_tbl, :down_tbl, 'PAUSED', :reason);
        paused_count := paused_count + 1;
    END FOR;

    -- Resume downstream tables where upstream is now healthy
    LET resume_cur CURSOR FOR
        SELECT DISTINCT d.UPSTREAM_TABLE, d.DOWNSTREAM_TABLE
        FROM TABLE_DEPENDENCIES d
        WHERE d.DOWNSTREAM_TABLE IN (SELECT DOWNSTREAM_TABLE FROM DOWNSTREAM_PAUSE_LOG WHERE ACTION = 'PAUSED')
          AND NOT EXISTS (
              SELECT 1 FROM SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS q
              WHERE q.TABLE_NAME = SPLIT_PART(d.UPSTREAM_TABLE, '.', 2)
                AND q.PASSED = FALSE
                AND q.SEVERITY IN ('CRITICAL', 'HIGH')
                AND q.REMEDIATION_STATUS NOT IN ('RESOLVED')
                AND q.CHECK_TIMESTAMP >= DATEADD('hour', -1, CURRENT_TIMESTAMP())
          );

    FOR rec IN resume_cur DO
        LET r_up VARCHAR := rec.UPSTREAM_TABLE;
        LET r_down VARCHAR := rec.DOWNSTREAM_TABLE;
        INSERT INTO DOWNSTREAM_PAUSE_LOG (UPSTREAM_TABLE, DOWNSTREAM_TABLE, ACTION, REASON)
        VALUES (:r_up, :r_down, 'RESUMED', 'Upstream issues resolved. Resuming downstream processing.');
        resumed_count := resumed_count + 1;
    END FOR;

    RETURN 'Dependency check complete. Paused: ' || :paused_count || ', Resumed: ' || :resumed_count;
END;

CALL SELF_HEALING_PIPELINE.OPERATIONS.CHECK_DEPENDENCIES();

-- View dependency status
SELECT 
    d.UPSTREAM_TABLE, d.DOWNSTREAM_TABLE, d.DEPENDENCY_TYPE,
    COALESCE(p.ACTION, 'RUNNING') AS CURRENT_STATUS,
    p.REASON,
    p.ACTION_TIMESTAMP
FROM TABLE_DEPENDENCIES d
LEFT JOIN (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY DOWNSTREAM_TABLE ORDER BY ACTION_TIMESTAMP DESC) AS rn
    FROM DOWNSTREAM_PAUSE_LOG
) p ON d.DOWNSTREAM_TABLE = p.DOWNSTREAM_TABLE AND p.rn = 1
ORDER BY d.UPSTREAM_TABLE;

## 14. Cost & Time Savings Tracker
Quantify how much the pipeline saves by calculating engineer-hours avoided and incident response costs.

In [ ]:
%%sql -r roi_data
USE SCHEMA SELF_HEALING_PIPELINE.OPERATIONS;

-- Cost configuration table
CREATE OR REPLACE TABLE COST_PARAMETERS (
    PARAM_NAME VARCHAR(100) PRIMARY KEY,
    PARAM_VALUE FLOAT,
    DESCRIPTION VARCHAR(500)
);

-- Industry-standard cost assumptions
INSERT INTO COST_PARAMETERS VALUES
    ('AVG_ENGINEER_HOURLY_RATE', 85.00, 'Average data engineer hourly cost (USD)'),
    ('AVG_MANUAL_FIX_MINUTES', 45, 'Average minutes to manually investigate + fix a data issue'),
    ('AVG_ESCALATION_MINUTES', 120, 'Average minutes for escalated issue resolution'),
    ('DOWNTIME_COST_PER_HOUR', 500, 'Cost of downstream data being unavailable per hour'),
    ('FALSE_POSITIVE_MINUTES', 15, 'Minutes wasted on a false alarm');

-- View: Calculate total savings
CREATE OR REPLACE VIEW PIPELINE_ROI AS
SELECT
    -- Counts
    (SELECT COUNT(*) FROM REMEDIATION_AUDIT_LOG WHERE SUCCESS = TRUE) AS auto_fixes_applied,
    (SELECT COUNT(*) FROM REMEDIATION_AUDIT_LOG WHERE SUCCESS = FALSE) AS escalations,
    (SELECT COUNT(*) FROM SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS WHERE PASSED = FALSE) AS total_incidents,
    
    -- Time saved
    (SELECT COUNT(*) FROM REMEDIATION_AUDIT_LOG WHERE SUCCESS = TRUE) 
        * (SELECT PARAM_VALUE FROM COST_PARAMETERS WHERE PARAM_NAME = 'AVG_MANUAL_FIX_MINUTES') 
        / 60.0 AS engineer_hours_saved,
    
    -- Cost saved from auto-fixes
    (SELECT COUNT(*) FROM REMEDIATION_AUDIT_LOG WHERE SUCCESS = TRUE) 
        * (SELECT PARAM_VALUE FROM COST_PARAMETERS WHERE PARAM_NAME = 'AVG_MANUAL_FIX_MINUTES') 
        / 60.0
        * (SELECT PARAM_VALUE FROM COST_PARAMETERS WHERE PARAM_NAME = 'AVG_ENGINEER_HOURLY_RATE')
        AS dollars_saved_fixes,
    
    -- Cost saved from preventing downtime (each auto-fix prevents ~30 min downtime)
    (SELECT COUNT(*) FROM REMEDIATION_AUDIT_LOG WHERE SUCCESS = TRUE) 
        * 0.5 -- 30 min downtime prevented per fix
        * (SELECT PARAM_VALUE FROM COST_PARAMETERS WHERE PARAM_NAME = 'DOWNTIME_COST_PER_HOUR')
        AS dollars_saved_downtime,
    
    -- Total ROI
    ((SELECT COUNT(*) FROM REMEDIATION_AUDIT_LOG WHERE SUCCESS = TRUE) 
        * (SELECT PARAM_VALUE FROM COST_PARAMETERS WHERE PARAM_NAME = 'AVG_MANUAL_FIX_MINUTES') 
        / 60.0
        * (SELECT PARAM_VALUE FROM COST_PARAMETERS WHERE PARAM_NAME = 'AVG_ENGINEER_HOURLY_RATE'))
    + ((SELECT COUNT(*) FROM REMEDIATION_AUDIT_LOG WHERE SUCCESS = TRUE) 
        * 0.5 * (SELECT PARAM_VALUE FROM COST_PARAMETERS WHERE PARAM_NAME = 'DOWNTIME_COST_PER_HOUR'))
    AS total_dollars_saved,

    -- Mean time to resolution
    (SELECT PARAM_VALUE FROM COST_PARAMETERS WHERE PARAM_NAME = 'AVG_MANUAL_FIX_MINUTES') AS manual_mttr_minutes,
    1.5 AS automated_mttr_minutes, -- pipeline resolves in ~90 seconds
    ROUND(((SELECT PARAM_VALUE FROM COST_PARAMETERS WHERE PARAM_NAME = 'AVG_MANUAL_FIX_MINUTES') - 1.5) 
        / (SELECT PARAM_VALUE FROM COST_PARAMETERS WHERE PARAM_NAME = 'AVG_MANUAL_FIX_MINUTES') * 100, 1) 
        AS mttr_improvement_pct;

SELECT * FROM PIPELINE_ROI;

In [ ]:
import pandas as pd

roi = roi_data.to_pandas() if not isinstance(roi_data, pd.DataFrame) else roi_data.copy()

if not roi.empty:
    r = roi.iloc[0]
    print("="*60)
    print("       PIPELINE ROI & COST SAVINGS REPORT")
    print("="*60)
    print(f"\n  INCIDENTS")
    print(f"  {'─'*40}")
    print(f"  Total incidents detected:    {int(r['TOTAL_INCIDENTS'])}")
    print(f"  Auto-fixed:                  {int(r['AUTO_FIXES_APPLIED'])}")
    print(f"  Escalated to humans:         {int(r['ESCALATIONS'])}")
    
    print(f"\n  TIME SAVINGS")
    print(f"  {'─'*40}")
    print(f"  Engineer hours saved:        {r['ENGINEER_HOURS_SAVED']:.1f} hours")
    print(f"  Manual MTTR:                 {r['MANUAL_MTTR_MINUTES']:.0f} min/incident")
    print(f"  Automated MTTR:              {r['AUTOMATED_MTTR_MINUTES']:.1f} min/incident")
    print(f"  MTTR improvement:            {r['MTTR_IMPROVEMENT_PCT']:.1f}%")
    
    print(f"\n  COST SAVINGS (USD)")
    print(f"  {'─'*40}")
    print(f"  Engineer time saved:         ${r['DOLLARS_SAVED_FIXES']:,.2f}")
    print(f"  Downtime prevented:          ${r['DOLLARS_SAVED_DOWNTIME']:,.2f}")
    print(f"  ──────────────────────────────────────")
    print(f"  TOTAL SAVED:                 ${r['TOTAL_DOLLARS_SAVED']:,.2f}")
    
    print(f"\n  PROJECTED ANNUAL SAVINGS")
    print(f"  {'─'*40}")
    # Assuming current rate continues
    daily_fixes = max(int(r['AUTO_FIXES_APPLIED']), 1)
    annual = r['TOTAL_DOLLARS_SAVED'] * 365 / max(daily_fixes, 1) * daily_fixes
    print(f"  At current rate:             ${annual:,.0f}/year")
    print("\n" + "="*60)

## 15. Human-in-the-Loop Approval System
For critical or low-confidence fixes, the pipeline can require human approval before executing. This adds a review queue with approve/reject functionality.

In [ ]:
%%sql -r approval_queue
USE SCHEMA SELF_HEALING_PIPELINE.OPERATIONS;

-- Approval queue for fixes that need human review
CREATE OR REPLACE TABLE APPROVAL_QUEUE (
    APPROVAL_ID NUMBER AUTOINCREMENT,
    CHECK_ID NUMBER,
    TABLE_NAME VARCHAR(200),
    CHECK_TYPE VARCHAR(100),
    SEVERITY VARCHAR(20),
    PROPOSED_ACTION VARCHAR(2000),
    ROOT_CAUSE VARCHAR(2000),
    CONFIDENCE_SCORE FLOAT,
    STATUS VARCHAR(50) DEFAULT 'PENDING_REVIEW', -- PENDING_REVIEW, APPROVED, REJECTED, EXPIRED
    SUBMITTED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    REVIEWED_BY VARCHAR(200),
    REVIEWED_AT TIMESTAMP_NTZ,
    REVIEW_NOTES VARCHAR(2000)
);

-- Procedure: Route low-confidence or critical fixes to approval queue
CREATE OR REPLACE PROCEDURE SELF_HEALING_PIPELINE.OPERATIONS.ROUTE_TO_APPROVAL()
RETURNS VARCHAR
LANGUAGE SQL
AS
BEGIN
    LET routed NUMBER := 0;

    -- Route items where confidence < 0.80 OR severity is CRITICAL
    INSERT INTO APPROVAL_QUEUE (CHECK_ID, TABLE_NAME, CHECK_TYPE, SEVERITY, PROPOSED_ACTION, ROOT_CAUSE, CONFIDENCE_SCORE)
    SELECT 
        c.CHECK_ID,
        q.TABLE_NAME,
        q.CHECK_TYPE,
        c.AI_SEVERITY,
        c.RECOMMENDED_ACTION,
        c.ROOT_CAUSE,
        c.CONFIDENCE_SCORE
    FROM ANOMALY_CLASSIFICATIONS c
    JOIN SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS q ON c.CHECK_ID = q.CHECK_ID
    WHERE (c.CONFIDENCE_SCORE < 0.80 OR c.AI_SEVERITY = 'CRITICAL')
      AND c.CHECK_ID NOT IN (SELECT CHECK_ID FROM APPROVAL_QUEUE)
      AND q.REMEDIATION_STATUS IN ('PENDING', 'IN_PROGRESS');

    SELECT COUNT(*) INTO :routed FROM APPROVAL_QUEUE WHERE STATUS = 'PENDING_REVIEW';
    RETURN :routed || ' items awaiting human review.';
END;

-- Procedure: Approve a fix (simulates the Streamlit button click)
CREATE OR REPLACE PROCEDURE SELF_HEALING_PIPELINE.OPERATIONS.APPROVE_FIX(P_APPROVAL_ID NUMBER, P_REVIEWER VARCHAR)
RETURNS VARCHAR
LANGUAGE SQL
AS
BEGIN
    UPDATE APPROVAL_QUEUE
    SET STATUS = 'APPROVED', REVIEWED_BY = :P_REVIEWER, REVIEWED_AT = CURRENT_TIMESTAMP()
    WHERE APPROVAL_ID = :P_APPROVAL_ID AND STATUS = 'PENDING_REVIEW';
    RETURN 'Approved. Fix will be applied on next remediation cycle.';
END;

-- Procedure: Reject a fix
CREATE OR REPLACE PROCEDURE SELF_HEALING_PIPELINE.OPERATIONS.REJECT_FIX(P_APPROVAL_ID NUMBER, P_REVIEWER VARCHAR, P_NOTES VARCHAR)
RETURNS VARCHAR
LANGUAGE SQL
AS
BEGIN
    UPDATE APPROVAL_QUEUE
    SET STATUS = 'REJECTED', REVIEWED_BY = :P_REVIEWER, REVIEWED_AT = CURRENT_TIMESTAMP(), REVIEW_NOTES = :P_NOTES
    WHERE APPROVAL_ID = :P_APPROVAL_ID AND STATUS = 'PENDING_REVIEW';
    RETURN 'Rejected. Issue will remain escalated.';
END;

-- Route current items and show queue
CALL SELF_HEALING_PIPELINE.OPERATIONS.ROUTE_TO_APPROVAL();

SELECT APPROVAL_ID, TABLE_NAME, CHECK_TYPE, SEVERITY, 
       PROPOSED_ACTION, CONFIDENCE_SCORE, STATUS, SUBMITTED_AT
FROM APPROVAL_QUEUE
ORDER BY SUBMITTED_AT DESC;

In [ ]:
%%sql -r approval_demo
-- Example: Approve fix #1 (simulate a human clicking 'Approve')
-- CALL SELF_HEALING_PIPELINE.OPERATIONS.APPROVE_FIX(1, 'NITHYASRI2006');

-- Example: Reject fix #2 with notes
-- CALL SELF_HEALING_PIPELINE.OPERATIONS.REJECT_FIX(2, 'NITHYASRI2006', 'Root cause seems incorrect, needs manual investigation');

SELECT 'Approval system ready. Use APPROVE_FIX() or REJECT_FIX() to review items.' AS status;

## 16. Multi-Source Ingestion Monitor
Extend monitoring beyond tables to track data from APIs, S3 stages, and streaming sources. Each source has health tracking and SLA enforcement.

In [ ]:
%%sql -r source_registry
USE SCHEMA SELF_HEALING_PIPELINE.MONITORING;

-- Data source registry: all ingestion sources tracked by the pipeline
CREATE OR REPLACE TABLE DATA_SOURCE_REGISTRY (
    SOURCE_ID NUMBER AUTOINCREMENT,
    SOURCE_NAME VARCHAR(200),
    SOURCE_TYPE VARCHAR(50), -- S3, API, KAFKA, DATABASE, SFTP
    CONNECTION_STRING VARCHAR(1000),
    TARGET_TABLE VARCHAR(500),
    EXPECTED_FREQUENCY_MINUTES NUMBER, -- how often data should arrive
    SLA_FRESHNESS_MINUTES NUMBER, -- max acceptable staleness
    IS_ACTIVE BOOLEAN DEFAULT TRUE,
    LAST_SUCCESSFUL_LOAD TIMESTAMP_NTZ,
    LAST_CHECK_TIMESTAMP TIMESTAMP_NTZ,
    CURRENT_STATUS VARCHAR(50) DEFAULT 'UNKNOWN', -- HEALTHY, DEGRADED, DOWN, UNKNOWN
    CONSECUTIVE_FAILURES NUMBER DEFAULT 0,
    CREATED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Source health history
CREATE OR REPLACE TABLE SOURCE_HEALTH_LOG (
    LOG_ID NUMBER AUTOINCREMENT,
    SOURCE_ID NUMBER,
    CHECK_TIMESTAMP TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    STATUS VARCHAR(50),
    LATENCY_SECONDS FLOAT,
    RECORDS_RECEIVED NUMBER,
    ERROR_MESSAGE VARCHAR(2000)
);

-- Register sample data sources
INSERT INTO DATA_SOURCE_REGISTRY 
    (SOURCE_NAME, SOURCE_TYPE, CONNECTION_STRING, TARGET_TABLE, EXPECTED_FREQUENCY_MINUTES, SLA_FRESHNESS_MINUTES, LAST_SUCCESSFUL_LOAD, CURRENT_STATUS)
VALUES
    ('Orders API', 'API', 'https://api.example.com/v2/orders', 'RAW.ORDERS_LANDING', 15, 30, DATEADD('minute', -10, CURRENT_TIMESTAMP()), 'HEALTHY'),
    ('Inventory S3 Feed', 'S3', 's3://data-lake/inventory/daily/', 'RAW.INVENTORY_LANDING', 1440, 1500, DATEADD('hour', -2, CURRENT_TIMESTAMP()), 'HEALTHY'),
    ('Payment Kafka Stream', 'KAFKA', 'kafka://broker:9092/payments', 'RAW.PAYMENTS_STREAM', 1, 5, DATEADD('minute', -3, CURRENT_TIMESTAMP()), 'HEALTHY'),
    ('Supplier SFTP', 'SFTP', 'sftp://supplier.partner.com/outbound/', 'RAW.SUPPLIER_UPDATES', 720, 780, DATEADD('hour', -8, CURRENT_TIMESTAMP()), 'HEALTHY'),
    ('Customer Reviews API', 'API', 'https://reviews.example.com/feed', 'RAW.CUSTOMER_REVIEWS', 60, 90, DATEADD('hour', -4, CURRENT_TIMESTAMP()), 'DEGRADED'),
    ('Clickstream Kafka', 'KAFKA', 'kafka://broker:9092/clicks', 'RAW.CLICKSTREAM', 1, 3, DATEADD('minute', -45, CURRENT_TIMESTAMP()), 'DOWN');

SELECT 'Data source registry populated' AS status;

In [ ]:
%%sql -r source_health
-- Procedure: Monitor all registered data sources for SLA violations
CREATE OR REPLACE PROCEDURE SELF_HEALING_PIPELINE.MONITORING.CHECK_SOURCE_HEALTH()
RETURNS VARCHAR
LANGUAGE SQL
AS
BEGIN
    LET checked NUMBER := 0;
    LET violations NUMBER := 0;

    LET cur CURSOR FOR
        SELECT SOURCE_ID, SOURCE_NAME, SOURCE_TYPE, TARGET_TABLE,
               SLA_FRESHNESS_MINUTES, LAST_SUCCESSFUL_LOAD, CONSECUTIVE_FAILURES
        FROM SELF_HEALING_PIPELINE.MONITORING.DATA_SOURCE_REGISTRY
        WHERE IS_ACTIVE = TRUE;

    FOR rec IN cur DO
        LET src_id NUMBER := rec.SOURCE_ID;
        LET sla_min NUMBER := rec.SLA_FRESHNESS_MINUTES;
        LET last_load TIMESTAMP_NTZ := rec.LAST_SUCCESSFUL_LOAD;
        LET failures NUMBER := rec.CONSECUTIVE_FAILURES;
        LET minutes_since FLOAT := TIMESTAMPDIFF('minute', :last_load, CURRENT_TIMESTAMP());
        LET new_status VARCHAR;
        LET new_failures NUMBER;

        IF (:minutes_since <= :sla_min) THEN
            new_status := 'HEALTHY';
            new_failures := 0;
        ELSEIF (:minutes_since <= :sla_min * 2) THEN
            new_status := 'DEGRADED';
            new_failures := :failures + 1;
            violations := violations + 1;
        ELSE
            new_status := 'DOWN';
            new_failures := :failures + 1;
            violations := violations + 1;
        END IF;

        UPDATE SELF_HEALING_PIPELINE.MONITORING.DATA_SOURCE_REGISTRY
        SET CURRENT_STATUS = :new_status,
            CONSECUTIVE_FAILURES = :new_failures,
            LAST_CHECK_TIMESTAMP = CURRENT_TIMESTAMP()
        WHERE SOURCE_ID = :src_id;

        INSERT INTO SELF_HEALING_PIPELINE.MONITORING.SOURCE_HEALTH_LOG (SOURCE_ID, STATUS, LATENCY_SECONDS)
        VALUES (:src_id, :new_status, :minutes_since * 60);

        checked := checked + 1;
    END FOR;

    RETURN 'Checked ' || :checked || ' sources. SLA violations: ' || :violations;
END;

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("CALL SELF_HEALING_PIPELINE.MONITORING.CHECK_SOURCE_HEALTH()").collect()

# Get source health data
sh = session.sql("""
    SELECT SOURCE_NAME, SOURCE_TYPE, TARGET_TABLE, CURRENT_STATUS, CONSECUTIVE_FAILURES,
           TIMESTAMPDIFF('minute', LAST_SUCCESSFUL_LOAD, CURRENT_TIMESTAMP()) AS MINUTES_SINCE_LAST_LOAD,
           SLA_FRESHNESS_MINUTES AS SLA_LIMIT
    FROM SELF_HEALING_PIPELINE.MONITORING.DATA_SOURCE_REGISTRY
    WHERE IS_ACTIVE = TRUE
    ORDER BY CASE CURRENT_STATUS WHEN 'DOWN' THEN 1 WHEN 'DEGRADED' THEN 2 ELSE 3 END
""").to_pandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MULTI-SOURCE INGESTION MONITOR', fontsize=14, fontweight='bold')

# Panel 1: Source status overview
ax1 = axes[0]
status_counts = sh['CURRENT_STATUS'].value_counts()
colors_map = {'HEALTHY': '#2ecc71', 'DEGRADED': '#f39c12', 'DOWN': '#e74c3c', 'UNKNOWN': '#95a5a6'}
colors = [colors_map.get(s, '#95a5a6') for s in status_counts.index]
ax1.bar(status_counts.index, status_counts.values, color=colors, edgecolor='white', width=0.6)
for i, (idx, val) in enumerate(zip(status_counts.index, status_counts.values)):
    ax1.text(i, val + 0.05, str(val), ha='center', fontweight='bold', fontsize=12)
ax1.set_title('Source Status Overview')
ax1.set_ylabel('Count')

# Panel 2: SLA compliance
ax2 = axes[1]
sources = sh['SOURCE_NAME'].tolist()
minutes = sh['MINUTES_SINCE_LAST_LOAD'].astype(float).tolist()
sla = sh['SLA_LIMIT'].astype(float).tolist()

x = range(len(sources))
ax2.barh(list(x), minutes, color=['#e74c3c' if m > s else '#2ecc71' for m, s in zip(minutes, sla)], height=0.5, label='Minutes since load')
ax2.scatter(sla, list(x), color='black', marker='|', s=200, linewidths=2, label='SLA Limit')
ax2.set_yticks(list(x))
ax2.set_yticklabels([s[:20] for s in sources], fontsize=8)
ax2.set_xlabel('Minutes')
ax2.set_title('Freshness vs SLA')
ax2.legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nSources: {len(sh)} total | "
      f"{(sh['CURRENT_STATUS']=='HEALTHY').sum()} healthy | "
      f"{(sh['CURRENT_STATUS']=='DEGRADED').sum()} degraded | "
      f"{(sh['CURRENT_STATUS']=='DOWN').sum()} down")

---
## 17. Complete Visual Dashboard
A comprehensive, single-view dashboard showing all pipeline metrics at a glance.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Fetch all dashboard data
health_df = session.sql("""
    SELECT COUNT(*) AS total, COUNT_IF(PASSED=TRUE) AS passed, COUNT_IF(PASSED=FALSE) AS failed,
           COUNT_IF(REMEDIATION_STATUS='RESOLVED') AS resolved,
           COUNT_IF(REMEDIATION_STATUS='ESCALATED') AS escalated,
           COUNT_IF(REMEDIATION_STATUS='PENDING') AS pending
    FROM SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS
""").to_pandas()

severity_df = session.sql("""
    SELECT SEVERITY, COUNT(*) AS cnt
    FROM SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS
    WHERE PASSED = FALSE
    GROUP BY SEVERITY
""").to_pandas()

patterns_df = session.sql("""
    SELECT TABLE_NAME, CHECK_TYPE, OCCURRENCE_COUNT, TREND, PATTERN_STATUS,
           PREDICTED_NEXT_OCCURRENCE
    FROM SELF_HEALING_PIPELINE.MONITORING.ANOMALY_PATTERNS
    ORDER BY OCCURRENCE_COUNT DESC
""").to_pandas()

sources_df = session.sql("""
    SELECT SOURCE_NAME, SOURCE_TYPE, CURRENT_STATUS,
           TIMESTAMPDIFF('minute', LAST_SUCCESSFUL_LOAD, CURRENT_TIMESTAMP()) AS MINS_STALE,
           SLA_FRESHNESS_MINUTES AS SLA
    FROM SELF_HEALING_PIPELINE.MONITORING.DATA_SOURCE_REGISTRY
    WHERE IS_ACTIVE = TRUE
    ORDER BY CASE CURRENT_STATUS WHEN 'DOWN' THEN 1 WHEN 'DEGRADED' THEN 2 ELSE 3 END
""").to_pandas()

audit_df = session.sql("""
    SELECT ACTION_TAKEN, SUCCESS, ACTION_TIMESTAMP
    FROM SELF_HEALING_PIPELINE.OPERATIONS.REMEDIATION_AUDIT_LOG
    ORDER BY ACTION_TIMESTAMP DESC LIMIT 10
""").to_pandas()

roi_df = session.sql("SELECT * FROM SELF_HEALING_PIPELINE.OPERATIONS.PIPELINE_ROI").to_pandas()

print("Data loaded successfully.")

---
## 17. Complete Visual Dashboard
A comprehensive, single-view dashboard showing all pipeline metrics at a glance.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Fetch all dashboard data
health_df = session.sql("""
    SELECT COUNT(*) AS total, COUNT_IF(PASSED=TRUE) AS passed, COUNT_IF(PASSED=FALSE) AS failed,
           COUNT_IF(REMEDIATION_STATUS='RESOLVED') AS resolved,
           COUNT_IF(REMEDIATION_STATUS='ESCALATED') AS escalated,
           COUNT_IF(REMEDIATION_STATUS='PENDING') AS pending
    FROM SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS
""").to_pandas()

severity_df = session.sql("""
    SELECT SEVERITY, COUNT(*) AS cnt
    FROM SELF_HEALING_PIPELINE.MONITORING.QUALITY_CHECK_RESULTS
    WHERE PASSED = FALSE
    GROUP BY SEVERITY
""").to_pandas()

patterns_df = session.sql("""
    SELECT TABLE_NAME, CHECK_TYPE, OCCURRENCE_COUNT, TREND, PATTERN_STATUS,
           PREDICTED_NEXT_OCCURRENCE
    FROM SELF_HEALING_PIPELINE.MONITORING.ANOMALY_PATTERNS
    ORDER BY OCCURRENCE_COUNT DESC
""").to_pandas()

sources_df = session.sql("""
    SELECT SOURCE_NAME, SOURCE_TYPE, CURRENT_STATUS,
           TIMESTAMPDIFF('minute', LAST_SUCCESSFUL_LOAD, CURRENT_TIMESTAMP()) AS MINS_STALE,
           SLA_FRESHNESS_MINUTES AS SLA
    FROM SELF_HEALING_PIPELINE.MONITORING.DATA_SOURCE_REGISTRY
    WHERE IS_ACTIVE = TRUE
    ORDER BY CASE CURRENT_STATUS WHEN 'DOWN' THEN 1 WHEN 'DEGRADED' THEN 2 ELSE 3 END
""").to_pandas()

audit_df = session.sql("""
    SELECT ACTION_TAKEN, SUCCESS, ACTION_TIMESTAMP
    FROM SELF_HEALING_PIPELINE.OPERATIONS.REMEDIATION_AUDIT_LOG
    ORDER BY ACTION_TIMESTAMP DESC LIMIT 10
""").to_pandas()

roi_df = session.sql("SELECT * FROM SELF_HEALING_PIPELINE.OPERATIONS.PIPELINE_ROI").to_pandas()

print("Data loaded successfully.")

In [ ]:
# === COMPREHENSIVE PIPELINE DASHBOARD ===
fig = plt.figure(figsize=(18, 22))
gs = GridSpec(5, 3, figure=fig, hspace=0.4, wspace=0.3)

fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('SELF-HEALING DATA PIPELINE — COMMAND CENTER', 
             fontsize=20, fontweight='bold', color='white', y=0.98)

# Color scheme
GREEN = '#00d2a0'
RED = '#ff4757'
YELLOW = '#ffa502'
BLUE = '#3742fa'
WHITE = '#ffffff'
GRAY = '#636e72'
DARK_BG = '#2d3436'

# ===== ROW 1: KPI CARDS =====
h = health_df.iloc[0]
total_checks = int(h['TOTAL'])
passed = int(h['PASSED'])
failed = int(h['FAILED'])
resolved = int(h['RESOLVED'])
health_pct = passed / total_checks * 100 if total_checks > 0 else 0
remediation_rate = resolved / max(failed, 1) * 100

# KPI 1: Health Score
ax_kpi1 = fig.add_subplot(gs[0, 0])
ax_kpi1.set_facecolor(DARK_BG)
ax_kpi1.text(0.5, 0.7, f'{health_pct:.0f}%', ha='center', va='center', 
             fontsize=42, fontweight='bold', color=GREEN if health_pct >= 80 else YELLOW if health_pct >= 60 else RED,
             transform=ax_kpi1.transAxes)
ax_kpi1.text(0.5, 0.25, 'HEALTH SCORE', ha='center', va='center', 
             fontsize=11, color=GRAY, transform=ax_kpi1.transAxes)
ax_kpi1.text(0.5, 0.1, f'{passed}/{total_checks} checks passed', ha='center', 
             fontsize=9, color=GRAY, transform=ax_kpi1.transAxes)
ax_kpi1.set_xlim(0,1); ax_kpi1.set_ylim(0,1)
ax_kpi1.axis('off')

# KPI 2: Auto-Fix Rate
ax_kpi2 = fig.add_subplot(gs[0, 1])
ax_kpi2.set_facecolor(DARK_BG)
ax_kpi2.text(0.5, 0.7, f'{remediation_rate:.0f}%', ha='center', va='center', 
             fontsize=42, fontweight='bold', color=GREEN if remediation_rate >= 80 else YELLOW,
             transform=ax_kpi2.transAxes)
ax_kpi2.text(0.5, 0.25, 'AUTO-FIX RATE', ha='center', va='center', 
             fontsize=11, color=GRAY, transform=ax_kpi2.transAxes)
ax_kpi2.text(0.5, 0.1, f'{resolved} of {failed} issues auto-resolved', ha='center', 
             fontsize=9, color=GRAY, transform=ax_kpi2.transAxes)
ax_kpi2.set_xlim(0,1); ax_kpi2.set_ylim(0,1)
ax_kpi2.axis('off')

# KPI 3: Cost Saved
ax_kpi3 = fig.add_subplot(gs[0, 2])
ax_kpi3.set_facecolor(DARK_BG)
if not roi_df.empty:
    saved = roi_df.iloc[0]['TOTAL_DOLLARS_SAVED']
    hours = roi_df.iloc[0]['ENGINEER_HOURS_SAVED']
else:
    saved = 0; hours = 0
ax_kpi3.text(0.5, 0.7, f'${saved:,.0f}', ha='center', va='center', 
             fontsize=42, fontweight='bold', color=GREEN, transform=ax_kpi3.transAxes)
ax_kpi3.text(0.5, 0.25, 'TOTAL SAVED', ha='center', va='center', 
             fontsize=11, color=GRAY, transform=ax_kpi3.transAxes)
ax_kpi3.text(0.5, 0.1, f'{hours:.1f} engineer-hours saved', ha='center', 
             fontsize=9, color=GRAY, transform=ax_kpi3.transAxes)
ax_kpi3.set_xlim(0,1); ax_kpi3.set_ylim(0,1)
ax_kpi3.axis('off')

# ===== ROW 2: SEVERITY + REMEDIATION STATUS =====
# Severity Distribution
ax_sev = fig.add_subplot(gs[1, 0:2])
ax_sev.set_facecolor(DARK_BG)
if not severity_df.empty:
    sev_order = ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']
    sev_colors = {'CRITICAL': RED, 'HIGH': '#e67e22', 'MEDIUM': YELLOW, 'LOW': GREEN}
    severity_df['sort'] = severity_df['SEVERITY'].map({s:i for i,s in enumerate(sev_order)})
    severity_df = severity_df.sort_values('sort')
    bars = ax_sev.bar(severity_df['SEVERITY'], severity_df['CNT'], 
                      color=[sev_colors.get(s, GRAY) for s in severity_df['SEVERITY']],
                      edgecolor='none', width=0.6)
    for bar, val in zip(bars, severity_df['CNT']):
        ax_sev.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                   str(int(val)), ha='center', color=WHITE, fontweight='bold')
ax_sev.set_title('FAILURES BY SEVERITY', color=WHITE, fontsize=11, fontweight='bold')
ax_sev.tick_params(colors=GRAY)
ax_sev.spines['bottom'].set_color(GRAY)
ax_sev.spines['left'].set_color(GRAY)
ax_sev.spines['top'].set_visible(False)
ax_sev.spines['right'].set_visible(False)

# Remediation Donut
ax_rem = fig.add_subplot(gs[1, 2])
ax_rem.set_facecolor(DARK_BG)
rem_data = [resolved, int(h['ESCALATED']), int(h['PENDING'])]
rem_labels = ['Resolved', 'Escalated', 'Pending']
rem_colors = [GREEN, RED, YELLOW]
rem_data_filtered = [(d, l, c) for d, l, c in zip(rem_data, rem_labels, rem_colors) if d > 0]
if rem_data_filtered:
    vals, labs, cols = zip(*rem_data_filtered)
    wedges, texts = ax_rem.pie(vals, colors=cols, startangle=90,
                               wedgeprops={'width': 0.4, 'edgecolor': DARK_BG})
    ax_rem.legend(labs, loc='lower center', fontsize=8, 
                 labelcolor=WHITE, frameon=False)
ax_rem.set_title('REMEDIATION STATUS', color=WHITE, fontsize=11, fontweight='bold')

# ===== ROW 3: SOURCE HEALTH =====
ax_src = fig.add_subplot(gs[2, :])
ax_src.set_facecolor(DARK_BG)
if not sources_df.empty:
    src_names = sources_df['SOURCE_NAME'].tolist()
    src_status = sources_df['CURRENT_STATUS'].tolist()
    mins_stale = sources_df['MINS_STALE'].astype(float).tolist()
    sla_vals = sources_df['SLA'].astype(float).tolist()
    
    y_pos = range(len(src_names))
    bar_colors = [GREEN if s == 'HEALTHY' else YELLOW if s == 'DEGRADED' else RED for s in src_status]
    ax_src.barh(list(y_pos), mins_stale, color=bar_colors, height=0.5, alpha=0.8)
    ax_src.scatter(sla_vals, list(y_pos), color=WHITE, marker='D', s=50, zorder=5, label='SLA Limit')
    ax_src.set_yticks(list(y_pos))
    ax_src.set_yticklabels(src_names, fontsize=9, color=WHITE)
    ax_src.set_xlabel('Minutes Since Last Load', color=GRAY, fontsize=9)
    ax_src.legend(loc='lower right', fontsize=8, labelcolor=WHITE, frameon=False)
ax_src.set_title('DATA SOURCE HEALTH — Freshness vs SLA', color=WHITE, fontsize=11, fontweight='bold')
ax_src.tick_params(colors=GRAY)
ax_src.spines['bottom'].set_color(GRAY)
ax_src.spines['left'].set_color(GRAY)
ax_src.spines['top'].set_visible(False)
ax_src.spines['right'].set_visible(False)

# ===== ROW 4: PATTERN ANALYSIS =====
ax_pat = fig.add_subplot(gs[3, :])
ax_pat.set_facecolor(DARK_BG)
if not patterns_df.empty:
    pat_labels = (patterns_df['CHECK_TYPE'] + '\n(' + patterns_df['TABLE_NAME'].str[:15] + ')').tolist()
    pat_counts = patterns_df['OCCURRENCE_COUNT'].astype(int).tolist()
    pat_trends = patterns_df['TREND'].tolist()
    trend_colors = {'WORSENING': RED, 'STABLE': YELLOW, 'IMPROVING': GREEN}
    colors = [trend_colors.get(t, GRAY) for t in pat_trends]
    
    bars = ax_pat.bar(range(len(pat_labels)), pat_counts, color=colors, edgecolor='none', width=0.6)
    ax_pat.set_xticks(range(len(pat_labels)))
    ax_pat.set_xticklabels(pat_labels, fontsize=8, color=WHITE, rotation=15, ha='right')
    for bar, val, trend in zip(bars, pat_counts, pat_trends):
        symbol = {'WORSENING': '▲', 'STABLE': '●', 'IMPROVING': '▼'}.get(trend, '?')
        ax_pat.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                   f'{val} {symbol}', ha='center', color=WHITE, fontsize=9)
    # Legend
    legend_patches = [mpatches.Patch(color=RED, label='Worsening ▲'),
                     mpatches.Patch(color=YELLOW, label='Stable ●'),
                     mpatches.Patch(color=GREEN, label='Improving ▼')]
    ax_pat.legend(handles=legend_patches, loc='upper right', fontsize=8, 
                 labelcolor=WHITE, frameon=False)
ax_pat.set_title('ANOMALY PATTERNS — Recurring Issues & Trends', color=WHITE, fontsize=11, fontweight='bold')
ax_pat.set_ylabel('Occurrences', color=GRAY, fontsize=9)
ax_pat.tick_params(colors=GRAY)
ax_pat.spines['bottom'].set_color(GRAY)
ax_pat.spines['left'].set_color(GRAY)
ax_pat.spines['top'].set_visible(False)
ax_pat.spines['right'].set_visible(False)

# ===== ROW 5: RECENT ACTIONS LOG =====
ax_log = fig.add_subplot(gs[4, :])
ax_log.set_facecolor(DARK_BG)
ax_log.axis('off')
ax_log.set_title('RECENT REMEDIATION ACTIONS', color=WHITE, fontsize=11, fontweight='bold', loc='left')

if not audit_df.empty:
    log_text = ''
    for i, row in audit_df.head(8).iterrows():
        icon = '✓' if row['SUCCESS'] else '✗'
        color_code = GREEN if row['SUCCESS'] else RED
        action = str(row['ACTION_TAKEN'])[:60]
        log_text += f"  {icon}  {action}\n"
    ax_log.text(0.02, 0.85, log_text, transform=ax_log.transAxes, fontsize=10,
               color=GREEN, fontfamily='monospace', verticalalignment='top')

plt.savefig('/tmp/pipeline_dashboard.png', dpi=100, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print("\nDashboard rendered. Saved to /tmp/pipeline_dashboard.png")